In [30]:
from pynq import Overlay
from pynq import allocate
from pynq import DefaultIP
import xrfdc
from xrfclk import set_ref_clks
import numpy as np

In [44]:
ol = Overlay("helloworld_wrapper.bit", ignore_version=True)
ol?

Type:            Overlay
String form:     <pynq.overlay.Overlay object at 0xffff740610c0>
File:            /usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/overlay.py
Docstring:      
Default documentation for overlay helloworld_wrapper.bit. The following
attributes are available on this overlay:

IP Blocks
----------
axi_fifo_mm_s        : pynq.overlay.DefaultIP
axi_dma              : pynq.lib.dma.DMA
usp_rf_data_converter_0 : xrfdc.RFdc
zynq_ultra_ps_e_0    : pynq.overlay.DefaultIP

Hierarchies
-----------
None

Interrupts
----------
None

GPIO Outputs
------------
None

Memories
------------
PSDDR                : Memory
Class docstring:
This class keeps track of a single bitstream's state and contents.

The overlay class holds the state of the bitstream and enables run-time
protection of bindings.

Our definition of overlay is: "post-bitstream configurable design".
Hence, this class must expose configurability through content discovery
and runtime protection.

The overl

In [32]:
dma = ol.axi_dma
rfdc = ol.usp_rf_data_converter_0

## Initialise RFDC LMK and LMX clocks

In [33]:
set_ref_clks(lmk_freq=122.88, lmx_freq=204.8) #LMK04832 LMX2594

## Set up RF Data Converters
Since were sending data out of the RF DAC and receiveing through the RF ADC, they need to be configured. <br>
Configure RF-ADC channel first.

BTW this is from https://github.com/strath-sdr/rfsoc_qsfp_offload/blob/master/boards/RFSoC4x2/rfsoc_qsfp_offload/drivers/overlay.py

In [34]:
def initialise_adc(tile, block, pll_freq=491.52, fs=4915.2, fc=0.0):
    """Initialise an ADC tile and block in bypass mode.
    """
    rfdc.adc_tiles[tile].DynamicPLLConfig(1, pll_freq, fs)
    rfdc.adc_tiles[tile].blocks[block].NyquistZone = 1
    rfdc.adc_tiles[tile].blocks[block].UpdateEvent(xrfdc.EVENT_MIXER)
    rfdc.adc_tiles[tile].SetupFIFO(True)

In [35]:
def initialise_dac(tile, block, pll_freq=491.52, fs=2457.60, fc=0.0):
        """Initialise a DAC tile and block in bypass mode.
            """
        rfdc.dac_tiles[tile].DynamicPLLConfig(1, pll_freq, fs)
        rfdc.dac_tiles[tile].blocks[block].NyquistZone = 1
        rfdc.dac_tiles[tile].blocks[block].MixerSettings['EventSource'] = xrfdc.EVNT_SRC_IMMEDIATE
        rfdc.dac_tiles[tile].SetupFIFO(True)

In [36]:
ADC_TILE = 0      # ADC Tile 226
ADC_BLOCK = 0       # ADC Block 0
ADC_SAMPLE_FREQUENCY = 4096  # MSps
ADC_PLL_FREQUENCY    = 204.8  # MHz
ADC_FC = -1228.8 # Centering around middle of sample rate

initialise_adc(tile=ADC_TILE,
              block=ADC_BLOCK,
              pll_freq=ADC_PLL_FREQUENCY,
              fs=ADC_SAMPLE_FREQUENCY,
              fc=ADC_FC)

Configure RF-DAC channel.

In [37]:
DAC_TILE = 0       # DAC Tile 228
DAC_BLOCK = 0       # DAC Block 0
DAC_SAMPLE_FREQUENCY = 4096  # MSps
DAC_PLL_FREQUENCY = 204.8   # MHz
DAC_FC = 0.0

initialise_dac(tile=DAC_TILE,
              block=DAC_BLOCK,
              pll_freq=DAC_PLL_FREQUENCY,
              fs=DAC_SAMPLE_FREQUENCY,
              fc=DAC_FC
             )

In [38]:
input_buffer = allocate(shape=(4000,), dtype=np.uint32)
output_buffer = allocate(shape=(4000,), dtype=np.uint32)

In [39]:
for i in range(4000):
    input_buffer[i] = 0;

In [40]:
dma.sendchannel.transfer(input_buffer)
dma.sendchannel.wait


<bound method _SDMAChannel.wait of <pynq.lib.dma._SDMAChannel object at 0xffff7620a230>>

NameError: name 'socket' is not defined

In [ ]:
start()

In [ ]:
stop()